# Free Embedding Server for RAG System

This notebook runs a **sentence-transformers** embedding server on Colab's free GPU
and exposes it via **ngrok** so your local RAG system can call it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhussein1504/rag-system/blob/main/colab_embedding_server.ipynb?flush_cache=true)


**Model**: `all-MiniLM-L6-v2` — 384-dim, very fast, great quality for RAG  
**Limit**: None — runs on Colab's free T4 GPU, fully local to Colab

In [1]:
!pip install -q sentence-transformers fastapi uvicorn pyngrok nest-asyncio

In [ ]:
import threading

import nest_asyncio
import uvicorn
from fastapi import FastAPI
from google.colab import userdata
from pydantic import BaseModel
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer, CrossEncoder

# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except Exception as e:
    if "already set" in str(e):
        print("ngrok auth token already configured, continuing...")
    else:
        raise e

# ── Model config ──────────────────────────────────────────────────────────────
# Change MODEL_NAME and DIMENSION together if you want a different model.
# Popular options:
#   all-MiniLM-L6-v2          → 384 dim  (fast, default)
#   all-mpnet-base-v2          → 768 dim  (better quality)
#   BAAI/bge-small-en-v1.5     → 384 dim  (great for retrieval)
#   BAAI/bge-base-en-v1.5      → 768 dim  (better retrieval)
MODEL_NAME = "BAAI/bge-base-en-v1.5"
RERANK_MODEL = "BAAI/bge-reranker-base"
DIMENSION  = 768
PORT       = 8000
# ─────────────────────────────────────────────────────────────────────────────

print(f"Loading model {MODEL_NAME!r} …")
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded. Embedding dimension: {DIMENSION}")

app = FastAPI(title="Colab Embedding Server")
reranker = CrossEncoder(RERANK_MODEL, max_length=512)


class EmbedRequest(BaseModel):
    texts: list[str]
    task_type: str = "RETRIEVAL_DOCUMENT"


class EmbedResponse(BaseModel):
    embeddings: list[list[float]]
    model: str
    dimension: int


class RerankRequest(BaseModel):
    query: str
    texts: list[str]


class RerankResponse(BaseModel):
    scores: list[float]
    model: str


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "dimension": DIMENSION}


@app.post("/embed", response_model=EmbedResponse)
def embed(req: EmbedRequest):
    if not req.texts:
        return EmbedResponse(embeddings=[], model=MODEL_NAME, dimension=DIMENSION)
    vectors = model.encode(
        req.texts,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()
    return EmbedResponse(embeddings=vectors, model=MODEL_NAME, dimension=DIMENSION)


@app.post("/rerank", response_model=RerankResponse)
def rerank(req: RerankRequest):
    pairs = [[req.query, text] for text in req.texts]
    scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)
    return RerankResponse(scores=[float(s) for s in scores], model=RERANK_MODEL)


nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print()
print("=" * 69)
print("> copy the next into your local .env:")
print()
print("EMBEDDING_BACKEND=remote")
print(f"EMBEDDING_MODEL_SIZE={DIMENSION}")
print(f"REMOTE_EMBEDDER_URL={public_url}")
print("=" * 69)

while True:
  continue

ngrok auth token already configured, continuing...
Loading model 'BAAI/bge-base-en-v1.5' …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 768


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

INFO:     Started server process [807]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



> copy the next into your local .env:

EMBEDDING_BACKEND=remote
EMBEDDING_MODEL_SIZE=768
REMOTE_EMBEDDER_URL=https://shifter-bottling-geranium.ngrok-free.dev
INFO:     196.148.28.23:0 - "POST /rerank HTTP/1.1" 404 Not Found
INFO:     196.148.28.23:0 - "POST /rerank HTTP/1.1" 404 Not Found


## ⚠️ Keep this notebook open!

The server runs as long as this Colab session is active.  
Colab free tier disconnects after **~90 minutes of inactivity** — just re-run Cell 3 if that happens (the ngrok URL will change, update your `.env`).

## Your local `.env` changes

```env
EMBEDDING_BACKEND=remote
EMBEDDING_MODEL_SIZE=384
REMOTE_EMBEDDER_URL=https://xxxx-xx-xx-xx-xx.ngrok-free.app
```

Then restart uvicorn (or it will hot-reload automatically).